# Nighttime Light Remote-Sensing Preprocessing

> This notebook calls the `functions/preprocessing.py` module to preprocess VNP46A1/A2 data.

**What it does**
- Stage 1: Extract nighttime light, observation angle, and quality-flag layers from HDF5 files, with optional clipping during extraction
- Stage 2: Generate standardized time-series text data, with automatic mosaicking for cross-tile study areas
- Optional: Clip the study area with a shapefile and automatically mosaic multiple tiles

In [ ]:
# Import the preprocessing module and reload it so edits take effect immediately
import sys, os
from importlib import import_module, reload

# Add the project path
cwd = os.getcwd()
project_root = cwd if os.path.basename(cwd) == 'ntl_prophet' else os.path.join(cwd, 'ntl_prophet')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Load and reload the module to pick up the latest changes
try:
    import functions.preprocessing as preprocessing_module
    reload(preprocessing_module)
except Exception:
    preprocessing_module = import_module('functions.preprocessing')
    reload(preprocessing_module)

# Import the functions used in this notebook
from functions.preprocessing import (
    stage1_extract_and_pair,
    stage2_generate_time_series,
    clip_rasters_by_shapefile,
    mosaic_tiles_by_date,
    complete_ntl_preprocessing_pipeline
)

print('✅ Preprocessing module loaded successfully')

## Parameters

Update the configuration below to match your local data paths.

In [ ]:
# ============== Configuration ==============
# Input paths
INPUT_VNP46A2_FOLDER = r'.\datasets\VNP46A2'   # VNP46A2 folder
INPUT_VNP46A1_FOLDER = r'.\datasets\VNP46A1'   # VNP46A1 folder

# Output paths
OUTPUT_BASE_FOLDER = r'.\processed'
FINAL_OUTPUT_FILE = r'.\processed\ntl_timeseries.txt'

# Zenith filling strategy: 'zero' or 'mean'
ZENITH_FILL_MODE = 'zero'

# ---- Date range filter ----
# Format: 'YYYY-MM-DD'; use None to disable filtering
DATE_START = '2023-01-01'
DATE_END   = '2025-01-01'

# ---- Tile filter ----
# Use None to process all tiles, or specify a tile list
# Supports strings such as 'h27v05' or tuples such as (27, 5)
TILE_FILTER = 'h21v05'

# ---- Clipping options ----
USE_SHAPE_CLIP = True  # Whether to clip with a shapefile
# Whether to clip using the shapefile bounding box; faster but less precise at the boundary
USE_SHAPE_CLIP_BY_BBOX = False
SHAPEFILE_PATH = r'.\datasets\shp\AOI.shp'
CLIP_PROCESSES = 4     # Number of parallel workers
CLIP_OVERWRITE = False
CLIP_WARP_OPTIONS = {'dstNodata': -9999}
CLIP_PARALLEL_MODE = 'auto'  # 'auto'|'process'|'thread'|'none'

# Whether to remove intermediate files
CLEANUP_INTERMEDIATE = True

print('🔧 Current configuration:')
print(f'   VNP46A2: {INPUT_VNP46A2_FOLDER}')
print(f'   VNP46A1: {INPUT_VNP46A1_FOLDER}')
print(f'   Output directory: {OUTPUT_BASE_FOLDER}')
print(f'   Output file: {FINAL_OUTPUT_FILE}')
print(f'   Zenith filling: {ZENITH_FILL_MODE}')
print(f"   Date range: {DATE_START or 'unlimited'} ~ {DATE_END or 'unlimited'}")
print(f"   Tile filter: {TILE_FILTER or 'all'}")
print(f'   Use clipping: {USE_SHAPE_CLIP}')
if USE_SHAPE_CLIP:
    print(f'   Clipping shapefile: {SHAPEFILE_PATH}')

## Run Preprocessing

Use `complete_ntl_preprocessing_pipeline()` to run the full workflow in one call.

In [ ]:
# Check input folders
vnp46a2_exists = os.path.exists(INPUT_VNP46A2_FOLDER)
vnp46a1_exists = os.path.exists(INPUT_VNP46A1_FOLDER)

print('📁 Folder check:')
print(f"   VNP46A2: {'✅' if vnp46a2_exists else '❌'} {INPUT_VNP46A2_FOLDER}")
print(f"   VNP46A1: {'✅' if vnp46a1_exists else '❌'} {INPUT_VNP46A1_FOLDER}")

if not vnp46a2_exists:
    raise FileNotFoundError('The VNP46A2 folder does not exist. Check the configured path.')

In [ ]:
# Run the full preprocessing workflow
result = complete_ntl_preprocessing_pipeline(
    input_vnp46a2_folder=INPUT_VNP46A2_FOLDER,
    input_vnp46a1_folder=INPUT_VNP46A1_FOLDER,
    output_base_folder=OUTPUT_BASE_FOLDER,
    final_output_file=FINAL_OUTPUT_FILE,
    cleanup_intermediate=CLEANUP_INTERMEDIATE,
    zenith_fill_mode=ZENITH_FILL_MODE,
    use_shape_clip=USE_SHAPE_CLIP,
    shapefile_path=SHAPEFILE_PATH if USE_SHAPE_CLIP else None,
    clip_processes=CLIP_PROCESSES,
    clip_overwrite=CLIP_OVERWRITE,
    clip_warp_options=CLIP_WARP_OPTIONS,
    clip_parallel_mode=CLIP_PARALLEL_MODE,
    use_shapefile_bbox=USE_SHAPE_CLIP_BY_BBOX,
    date_start=DATE_START,
    date_end=DATE_END,
    tile_filter=TILE_FILTER
)

print('✅ Preprocessing completed!')
print(f'   Output file: {FINAL_OUTPUT_FILE}')